# ReGlyco Demo: Glycosylation-Aware Binder Design Filtering on rEPO

Generate candidate binders against recombinant erythropoietin, then remove complexes that are incompatible with glycosylation before experimental testing.

The notebook goes in one pass:

1. Load the bundled rEPO target in **quick demo** mode, or switch to **advanced** mode for your own glycoprotein.
2. Define a candidate interaction patch on the protein surface.
3. Generate a small set of candidate mini-binders against the protein-only target.
4. Re-score the resulting complexes with ReGlyco and triage them into **Pass**, **Borderline**, or **Fail**.

This notebook uses the numbering in `inputs/REPO.pdb` throughout.

| Site used in this notebook | Glycan | Note |
| --- | --- | --- |
| **A24** | **G52258NG** | shifted N-linked site corresponding to hEPO N51 |
| **A38** | **G6878EM** | shifted N-linked site corresponding to hEPO N65 |
| **A83** | **G63547QM** | shifted N-linked site corresponding to hEPO N110 |
| **A126** | **G722008IY** | O-glycosylation site used in this demo |


In [ ]:
#@title Colab setup (GPU required for binder generation) { display-mode: "form" }

import os
from pathlib import Path

os.environ.setdefault('CCD_MIRROR_PATH', '')
os.environ.setdefault('PDB_MIRROR_PATH', '')

READY = Path('FOUNDRY_READY')

if not READY.exists():
    print('Installing rc-foundry (includes AtomWorks + RFD3 runtime)...')
    os.system('pip uninstall -y torchvision >/dev/null 2>&1')
    os.system("pip install -q 'rc-foundry[all]'")
    os.system('foundry install rfd3')
    READY.write_text('ok\n')
    print('Done.')
else:
    print('rc-foundry already installed (FOUNDRY_READY present).')

try:
    import torch
    if torch.cuda.is_available():
        print(f'CUDA available: {torch.cuda.device_count()} GPU(s)')
    else:
        print('CUDA not available (binder generation will be slow or unavailable).')
except Exception:
    pass


In [ ]:
#@title Setup (run once) { display-mode: "form" }

import os
import subprocess
import sys
from pathlib import Path
from IPython.display import HTML, display

REPO_URL = 'https://github.com/Ojas-Singh/GlycoShape-Resources.git'
REPO_NAME = 'GlycoShape-Resources'


def _find_repo_root() -> Path | None:
    candidates = [
        Path('.').resolve(),
        Path('/content') / REPO_NAME,
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / 'scripts' / 'paper_demo_helpers.py').exists() and (candidate / 'inputs' / 'REPO.pdb').exists():
            return candidate
    return None


repo_root = _find_repo_root()
if repo_root is None:
    try:
        import google.colab  # type: ignore
        _ = google.colab
        repo_root = Path('/content') / REPO_NAME
        if not repo_root.exists():
            print(f'Cloning {REPO_URL} into {repo_root} ...')
            subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(repo_root)])
    except Exception as exc:
        raise ModuleNotFoundError(
            'Could not locate the repo assets needed by this notebook. In Colab, rerun this cell so the repo can be cloned into /content.'
        ) from exc

if repo_root is None or not repo_root.exists():
    raise ModuleNotFoundError('Repo root could not be resolved.')

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Using repo root:', repo_root)

from scripts.paper_demo_helpers import (
    REPO_DEFAULT_GLYCANS,
    build_3dmol_gallery_html,
    build_remote_molstar_gallery_html,
    build_protein_only_structures,
    build_target_structures,
    clone_entries,
    generate_candidate_binders,
    load_target_context,
    parse_custom_glycan_json,
    prepare_patch,
    render_entry_table,
    render_patch_summary,
    run_compatibility_analysis,
)


## 1) Quick demo setup

The public notebook opens in **quick demo** mode by default. Leave the defaults unchanged for the paper-ready rEPO example, or switch to advanced mode to upload a different target, provide your own glycan list, or change how the interaction patch is selected.


In [ ]:
#@title Demo configuration { display-mode: "form" }

DEMO_MODE = "quick"  #@param ["quick", "advanced"]
TARGET_SOURCE = "repo"  #@param ["repo", "upload"]
PATCH_MODE = "preset"  #@param ["preset", "scan", "manual"]
GLYCAN_SOURCE = "repo_default"  #@param ["repo_default", "scan", "custom_json"]
N_BINDERS = 3  #@param {type:"slider", min:1, max:5, step:1}
BINDER_LENGTH = 100  #@param {type:"slider", min:80, max:130, step:5}
USE_COMPLEX_SCAN = True  #@param {type:"boolean"}
CUSTOM_TARGET_PDB_PATH = ""  #@param {type:"string"}
CUSTOM_PATCH_RESIDUES = ""  #@param {type:"string"}
CUSTOM_GLYCAN_JSON = '[{"chain":"A","resi":24,"glycan":"G52258NG"},{"chain":"A","resi":38,"glycan":"G6878EM"}]'  #@param {type:"string"}

resolved_target_source = 'repo' if DEMO_MODE == 'quick' else TARGET_SOURCE
resolved_patch_mode = PATCH_MODE
resolved_glycan_source = GLYCAN_SOURCE

if resolved_target_source != 'repo' and resolved_patch_mode == 'preset':
    resolved_patch_mode = 'scan'

if resolved_target_source != 'repo' and resolved_glycan_source == 'repo_default':
    resolved_glycan_source = 'custom_json' if (CUSTOM_GLYCAN_JSON or '').strip() else 'scan'

CONFIG = {
    'demo_mode': DEMO_MODE,
    'target_source': resolved_target_source,
    'patch_mode': resolved_patch_mode,
    'glycan_source': resolved_glycan_source,
    'n_binders': int(N_BINDERS),
    'binder_length': int(BINDER_LENGTH),
    'use_complex_scan': bool(USE_COMPLEX_SCAN),
    'custom_target_pdb_path': str(CUSTOM_TARGET_PDB_PATH).strip(),
    'custom_patch_residues': str(CUSTOM_PATCH_RESIDUES).strip(),
    'custom_glycan_json': str(CUSTOM_GLYCAN_JSON).strip(),
}

summary_html = f"""
<div style="font-family:system-ui, -apple-system, Segoe UI, Roboto, Arial; font-size:14px; border:1px solid #eee; border-radius:12px; padding:14px 16px; background:#fafafa;">
  <div style="font-weight:700; margin-bottom:8px;">Resolved notebook configuration</div>
  <div style="display:grid; grid-template-columns:repeat(2, minmax(0, 1fr)); gap:8px 18px;">
    <div><strong>Mode:</strong> {CONFIG['demo_mode']}</div>
    <div><strong>Target source:</strong> {CONFIG['target_source']}</div>
    <div><strong>Patch mode:</strong> {CONFIG['patch_mode']}</div>
    <div><strong>Glycan source:</strong> {CONFIG['glycan_source']}</div>
    <div><strong>Candidate binders:</strong> {CONFIG['n_binders']}</div>
    <div><strong>Binder length:</strong> {CONFIG['binder_length']} aa</div>
    <div><strong>Re-scan complexes:</strong> {"yes" if CONFIG['use_complex_scan'] else "no"}</div>
    <div><strong>Custom patch string:</strong> {CONFIG['custom_patch_residues'] or "not provided"}</div>
  </div>
</div>
"""
display(HTML(summary_html))


## 2) Load rEPO target and glycosylation context

The quick demo uses the bundled recombinant EPO structure in `inputs/REPO.pdb`. The glycosylation map is precomputed for this bundled target so the demo stays reproducible, while advanced mode can switch to an uploaded PDB and a user-provided glycan list.


In [ ]:
#@title Load the target and bundled glycosylation context

TARGET_STATE = load_target_context(CONFIG)
display(HTML(render_entry_table(
    'Configured glycosylation map',
    TARGET_STATE['configured_glycan_entries'],
    empty_text='The glycan map will be populated later by scan mode.',
)))

note_html = f"""
<div style="font-family:system-ui, -apple-system, Segoe UI, Roboto, Arial; font-size:13px; color:#444; margin:10px 0 0 0;">
  Loaded target: <strong>{TARGET_STATE['target_name']}</strong><br/>
  Bundled rEPO target: <strong>{"yes" if TARGET_STATE['target_is_repo'] else "no"}</strong><br/>
  Notebook glycosylation sites: <strong>A24, A38, A83, A126</strong>
</div>
"""
display(HTML(note_html))


In [ ]:
#@title Visualize the target, glycosylation sites, and the demo patch preview

display(HTML(build_3dmol_gallery_html(build_target_structures(TARGET_STATE), height=560)))


## 3) Define the candidate interaction patch

For the quick rEPO demo, the notebook uses a deterministic patch derived from the bundled example complex in `inputs/REPO_complex.pdb`. Advanced mode can switch to a scan-derived hotspot patch or a manual residue list.


In [ ]:
#@title Resolve the design patch and optional ReGlyco scan

PATCH_STATE = prepare_patch(CONFIG, TARGET_STATE)
display(HTML(render_patch_summary(PATCH_STATE)))
display(HTML(render_entry_table(
    'Glycosylation sites used for downstream filtering',
    PATCH_STATE['configured_glycan_entries'],
    empty_text='No glycosylation sites selected yet.',
)))
if PATCH_STATE['hotspots_pdb_url']:
    display(HTML(
        f"<div style='font-family:system-ui, -apple-system, Segoe UI, Roboto, Arial; font-size:13px; color:#444; margin-top:10px;'>Hotspot file: <a href='{PATCH_STATE['hotspots_pdb_url']}' target='_blank'>{PATCH_STATE['hotspots_pdb_url']}</a></div>"
    ))


## 4) Generate candidate mini-binders

Binder generation runs against the protein-only target and the selected interaction patch. The defaults are intentionally small for Colab: three candidate binders with a target length of about 100 amino acids.


In [ ]:
#@title RFD3 binder generation on the selected rEPO patch { display-mode: "form" }

GENERATION_STATE = generate_candidate_binders(CONFIG, TARGET_STATE, PATCH_STATE)


## 5) Inspect the protein-only complexes first

At this stage the candidate complexes are judged only from the protein backbone and interface geometry. This is the point where standard binder-design workflows would often stop.


In [ ]:
#@title Protein-only inspection of the generated complexes

if not GENERATION_STATE['generated_complex_pdb_paths']:
    display(HTML("<div style='font-family:system-ui, -apple-system, Segoe UI, Roboto, Arial; padding:10px;'>No generated complexes found. Run the binder generation cell first.</div>"))
else:
    display(HTML(build_3dmol_gallery_html(build_protein_only_structures(GENERATION_STATE), height=560)))


## 6) Run the ReGlyco filter on each binder complex

This step re-scores each candidate complex against the glycosylation map and classifies the result:

- **Pass**: no clashes after the first ReGlyco optimization pass.
- **Borderline**: the complex only clears after rotamer-enabled local adjustment.
- **Fail**: clashes persist even after the rotamer-enabled pass.


In [ ]:
#@title ReGlyco compatibility analysis: Pass / Borderline / Fail

ANALYSIS_STATE = run_compatibility_analysis(CONFIG, PATCH_STATE, GENERATION_STATE)
display(HTML(ANALYSIS_STATE['summary_html']))


In [ ]:
#@title Inspect the glyco-aware triage results

if not ANALYSIS_STATE['remote_structures']:
    display(HTML("<div style='font-family:system-ui, -apple-system, Segoe UI, Roboto, Arial; padding:10px;'>No compatibility-analysis results found. Run the ReGlyco compatibility cell first.</div>"))
else:
    display(HTML(build_remote_molstar_gallery_html(ANALYSIS_STATE['remote_structures'], height=660)))


## 7) Advanced custom target mode

To use your own glycoprotein target:

1. Switch `DEMO_MODE` to `advanced`.
2. Set `TARGET_SOURCE` to `upload` and provide a local path or upload a PDB when prompted.
3. Choose `GLYCAN_SOURCE = "custom_json"` and provide a small JSON list such as `[{"chain":"A","resi":24,"glycan":"G52258NG"}]`, or use `scan` to start from ReGlyco-detected sites.
4. Choose `PATCH_MODE = "manual"` with a string like `A24,A38,A83` or `PATCH_MODE = "scan"` for hotspot discovery.
